# JailbreakBench Colab Driver

Use this notebook on **Google Colab only**.

Colab's default notebook kernel is Python 3.12, but this project requires Python `<3.12` because of `jailbreakbench`. Colab also does not reliably let you switch to a custom kernel.

This notebook works around that by:
1. creating a Python 3.11 virtualenv
2. installing the repo into that venv
3. running the same `scripts/*.py` entrypoints through the venv interpreter

The repo code stays in `src/` and `scripts/` — this notebook is only a driver.

For local Jupyter with a Python 3.11 kernel, use [`reproduce_jailbreakbench.ipynb`](reproduce_jailbreakbench.ipynb) instead.

**Before running:** Runtime → Change runtime type → **GPU** (A100 recommended for Vicuna-13B fp16).

In [ ]:
!nvidia-smi

## Bootstrap — clone repo, create Python 3.11 venv, install project

Run this once per Colab session.

In [ ]:
import os
from pathlib import Path

REPO = Path("/content/dsc-291-trustworthy-project")
VENV = Path("/content/.venv311")
PY = VENV / "bin" / "python"

if not REPO.exists():
    !git clone https://github.com/ZachLiu519/dsc-291-trustworthy-project.git {REPO}

%cd {REPO}

!apt-get update -qq
!apt-get install -qq python3.11 python3.11-venv python3.11-dev

if not PY.exists():
    !python3.11 -m venv {VENV}

!{PY} -m pip install --upgrade pip
!{PY} -m pip install -e "{REPO}[dev,vllm]"

import subprocess
subprocess.run(
    [str(PY), "-c", "import jbb_repro; print('venv import ok:', jbb_repro.__file__)"],
    cwd=REPO,
    check=True,
)

## Secrets

Fill in your keys below, then run the cell. This writes `/content/.env`, which the scripts load automatically.

In [ ]:
OPENAI_API_KEY = "sk-..."
HF_TOKEN = "hf_..."

with open("/content/.env", "w", encoding="utf-8") as handle:
    handle.write(f"OPENAI_API_KEY={OPENAI_API_KEY}\n")
    handle.write(f"HF_TOKEN={HF_TOKEN}\n")

print("Wrote /content/.env")

## Driver helper

All experiment cells call `run(...)`, which executes scripts with the Python 3.11 venv.

In [ ]:
import os
import subprocess
from pathlib import Path

import pandas as pd

REPO = Path("/content/dsc-291-trustworthy-project")
PY = "/content/.venv311/bin/python"
ENV = {**os.environ, "PYTHONPATH": str(REPO / "src")}
CONFIGS = REPO / "configs"
OUTPUTS = REPO / "outputs"
REPORTS = REPO / "reports"

LIMIT = 2  # smoke test; set to None for full reproduction


def run(*args: str) -> None:
    subprocess.run([PY, *args], cwd=REPO, env=ENV, check=True)


def read_summary(relative_path: str) -> pd.DataFrame:
    return pd.read_csv(REPO / relative_path)


pd.set_option("display.max_colwidth", 120)
print(f"Repo: {REPO}")
print(f"Python: {PY}")
print(f"LIMIT: {LIMIT}")

## Step 0 — Verify wiring (dry run)

Build sampled behaviors and PAIR/GCG prompts without loading Vicuna.

In [ ]:
run(
    "scripts/run_vllm_local.py",
    "--config", "configs/vicuna_vllm.yaml",
    "--limit", "4",
    "--dry-run",
)

list((OUTPUTS / "vicuna_vllm_jbb_subset").glob("*"))

## Step 1 — Vicuna-13B harmful attacks (PAIR + GCG)

In [ ]:
args = ["scripts/run_vllm_local.py", "--config", "configs/vicuna_vllm.yaml"]
if LIMIT is not None:
    args.extend(["--limit", str(LIMIT)])
run(*args)

list((OUTPUTS / "vicuna_vllm_jbb_subset").glob("responses.jsonl"))

## Step 2 — Score Vicuna harmful responses

Primary judge: Llama-Guard-2. Heuristic scoring is a sensitivity check.

In [ ]:
run(
    "scripts/score_llamaguard.py",
    "--responses", "outputs/vicuna_vllm_jbb_subset/responses.jsonl",
)
run(
    "scripts/score_outputs.py",
    "--responses", "outputs/vicuna_vllm_jbb_subset/responses.jsonl",
)

display(read_summary("outputs/vicuna_vllm_jbb_subset/asr_summary_llamaguard.csv"))
display(read_summary("outputs/vicuna_vllm_jbb_subset/asr_summary.csv"))

## Step 3 — Vicuna-13B benign behaviors

In [ ]:
args = ["scripts/run_vllm_benign.py", "--config", "configs/vicuna_benign_vllm.yaml"]
if LIMIT is not None:
    args.extend(["--limit", str(LIMIT)])
run(*args)

run(
    "scripts/score_outputs.py",
    "--responses", "outputs/vicuna_benign_vllm_jbb_subset/responses.jsonl",
    "--benign",
)
run(
    "scripts/score_llamaguard.py",
    "--responses", "outputs/vicuna_benign_vllm_jbb_subset/responses.jsonl",
)

display(read_summary("outputs/vicuna_benign_vllm_jbb_subset/refusal_summary.csv"))
display(read_summary("outputs/vicuna_benign_vllm_jbb_subset/asr_summary_llamaguard.csv"))

## Step 4 — Dictionary-filter defense

In [ ]:
args = [
    "scripts/run_vllm_local.py",
    "--config", "configs/vicuna_dictionary_filter_vllm.yaml",
    "--defense", "dictionary_filter",
]
if LIMIT is not None:
    args.extend(["--limit", str(LIMIT)])
run(*args)

run(
    "scripts/score_llamaguard.py",
    "--responses", "outputs/vicuna_dictionary_filter_vllm_jbb_subset/responses.jsonl",
)

display(read_summary("outputs/vicuna_dictionary_filter_vllm_jbb_subset/asr_summary_llamaguard.csv"))

## Step 5 — GPT-4o-mini on the same attack prompts

In [ ]:
args = [
    "scripts/run_openai_model.py",
    "--config", "configs/gpt4o_mini_jbb.yaml",
    "--prompts", "outputs/vicuna_vllm_jbb_subset/attack_prompts.jsonl",
]
if LIMIT is not None:
    args.extend(["--limit", str(LIMIT)])
run(*args)

run(
    "scripts/score_llamaguard.py",
    "--responses", "outputs/gpt4o_mini_jbb_subset/responses.jsonl",
)
run(
    "scripts/score_outputs.py",
    "--responses", "outputs/gpt4o_mini_jbb_subset/responses.jsonl",
)

display(read_summary("outputs/gpt4o_mini_jbb_subset/asr_summary_llamaguard.csv"))
display(read_summary("outputs/gpt4o_mini_jbb_subset/asr_summary.csv"))

## Step 6 — Qualitative examples and paper comparison

Paper Vicuna harmful ASR: PAIR 69%, GCG 80%.

In [ ]:
run(
    "scripts/extract_qualitative_examples.py",
    "--scored", "outputs/vicuna_vllm_jbb_subset/responses_llamaguard_scored.csv",
    "--output", "reports/qualitative_examples_vicuna.md",
)
run(
    "scripts/extract_qualitative_examples.py",
    "--scored", "outputs/gpt4o_mini_jbb_subset/responses_llamaguard_scored.csv",
    "--output", "reports/qualitative_examples_gpt4o_mini.md",
)

vicuna_summary = read_summary("outputs/vicuna_vllm_jbb_subset/asr_summary_llamaguard.csv")
paper_comparison = pd.DataFrame(
    [
        {"attack": "PAIR", "paper_vicuna_asr": 0.69, "our_vicuna_asr": None},
        {"attack": "GCG", "paper_vicuna_asr": 0.80, "our_vicuna_asr": None},
    ]
)
for method in ("PAIR", "GCG"):
    row = vicuna_summary[vicuna_summary["method"] == method]
    if not row.empty:
        paper_comparison.loc[paper_comparison["attack"] == method, "our_vicuna_asr"] = float(
            row.iloc[0]["attack_success_rate"]
        )

display(paper_comparison)
print("Reports written under reports/qualitative_examples_*.md")

## Step 7 — Generate charts

In [ ]:
run("scripts/generate_charts.py", "--repo-root", str(REPO))

from IPython.display import Image, display

for name in sorted((REPORTS / "figures").glob("*.png")):
    print(name.name)
    display(Image(filename=str(name)))